In [ ]:
import numpy as np
import pandas as pd

# 1. 데이터 생성 핵심 함수 (기준 파일 로직 동일)
def generate_raw_items(n_students, n_items, target_mean, target_std, min_val, max_val, is_sum=False):
    scores = np.random.randn(n_students)
    scores = (scores - np.mean(scores)) / np.std(scores) * target_std + target_mean
    
    data = np.zeros((n_students, n_items))
    for i in range(n_students):
        target_sum = scores[i] if is_sum else scores[i] * n_items
        row = np.random.rand(n_items) + 0.5 
        row = (row / np.sum(row)) * target_sum
        row = np.clip(np.round(row), min_val, max_val)
        
        diff = int(round(target_sum)) - int(np.sum(row))
        attempts = 0
        while diff != 0 and attempts < 1000:
            idx = np.random.randint(n_items)
            if diff > 0 and row[idx] < max_val:
                row[idx] += 1
                diff -= 1
            elif diff < 0 and row[idx] > min_val:
                row[idx] -= 1
                diff += 1
            attempts += 1
        data[i] = row
    return data.astype(int)

# 2. 파라미터 설정 및 데이터 생성 (is_sum=False)
n_per_group = 18
n_items = 15

sdl_pre_exp = generate_raw_items(n_per_group, n_items, 5.15, 0.81, 1, 9, is_sum=False)
sdl_post_exp = generate_raw_items(n_per_group, n_items, 6.34, 0.82, 1, 9, is_sum=False)
sdl_pre_ctrl = generate_raw_items(n_per_group, n_items, 5.23, 0.80, 1, 9, is_sum=False)
sdl_post_ctrl = generate_raw_items(n_per_group, n_items, 5.41, 0.84, 1, 9, is_sum=False)

# 3. 데이터프레임 구축 함수
def build_sdl_df(group_name, sdl_pre, sdl_post, start_id):
    df = pd.DataFrame()
    df['Student_ID'] = [f'{group_name[0].upper()}{str(i).zfill(2)}' for i in range(start_id, start_id + n_per_group)]
    df['Group'] = group_name
    
    for i in range(n_items):
        df[f'SDL_Pre_Q{i+1}'] = sdl_pre[:, i]
        df[f'SDL_Post_Q{i+1}'] = sdl_post[:, i]
    
    # 학생별 평균 컬럼 추가 (검증용)
    df['SDL_Pre_Avg'] = df[[f'SDL_Pre_Q{i+1}' for i in range(n_items)]].mean(axis=1)
    df['SDL_Post_Avg'] = df[[f'SDL_Post_Q{i+1}' for i in range(n_items)]].mean(axis=1)
    return df

# 데이터 통합 및 저장
df_final_sdl = pd.concat([
    build_sdl_df('Experimental', sdl_pre_exp, sdl_post_exp, 1),
    build_sdl_df('Control', sdl_pre_ctrl, sdl_post_ctrl, 1)
], ignore_index=True)

df_final_sdl.to_csv('simulated_sdl_data.csv', index=False)

# 4. 검증 (기술통계량 확인)
print("=== 생성 데이터 검증 (SDL 문항 평균의 평균) ===")
print(df_final_sdl.groupby('Group')[['SDL_Pre_Avg', 'SDL_Post_Avg']].agg(['mean', 'std']).round(2))

✅ SDL 데이터 생성 완료! 파일 경로: ./sdl/simulated_sdl_data.csv
📊 데이터 형태: (36, 32) (36명, 32개 컬럼)
